<a href="https://colab.research.google.com/github/Yonaki97/DataScience/blob/main/Usecase02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Sep 30 11:20:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 140 not upgraded.


In [ ]:
!pip install -q ollama pymupdf numpy pandas openpyxl

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
ollama_server = subprocess.Popen(["ollama", "serve"])

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama pull qwen3-embedding:4b
!ollama pull qwen3.5:4b
!ollama serve > ollama.log 2>&1

In [ ]:
!curl http://127.0.0.1:11434

Ollama is running

In [ ]:
!ollama list

NAME                  ID              SIZE      MODIFIED               
qwen3.5:4b            2a654d98e6fb    3.4 GB    Less than a second ago    
qwen3-embedding:4b    df5bd2e3c74c    2.5 GB    Less than a second ago    


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import os
import sys

drive.mount("/content/drive", force_remount=True)

my_drive = Path("/content/drive/MyDrive")
zip_file = my_drive / "Project-UseCase.zip"
extract_dir = my_drive / "Project-UseCase_Gemini"

if not zip_file.is_file():
    raise FileNotFoundError(f"ZIP tidak ditemukan: {zip_file}")

# Ekstrak hanya jika belum pernah diekstrak atau pipeline.py belum ada.
pipeline_matches = list(extract_dir.rglob("pipeline.py")) if extract_dir.exists() else []

if not pipeline_matches:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    pipeline_matches = list(extract_dir.rglob("pipeline.py"))

if not pipeline_matches:
    raise FileNotFoundError(
        f"pipeline.py tidak ditemukan di dalam ZIP setelah diekstrak ke {extract_dir}"
    )

pipeline_file = pipeline_matches[0]
project_src = pipeline_file.parent
project_root = project_src.parent

print("ZIP          :", zip_file)
print("Folder proyek:", project_root)
print("pipeline.py  :", pipeline_file)

os.chdir(project_src)
sys.path.insert(0, str(project_src))

Mounted at /content/drive
ZIP          : /content/drive/MyDrive/Project-UseCase.zip
Folder proyek: /content/drive/MyDrive/Project-UseCase_Gemini/Project-UseCase
pipeline.py  : /content/drive/MyDrive/Project-UseCase_Gemini/Project-UseCase/src/pipeline.py


In [ ]:
%pip install -q streamlit

In [ ]:
import subprocess
import sys
import time
import urllib.request
from google.colab import output
from IPython.display import HTML, display

if not app_file.is_file():
    raise FileNotFoundError(f"app.py tidak ditemukan: {app_file}")

# Mulai Streamlit jika belum berjalan
if "streamlit_proc" not in globals() or streamlit_proc.poll() is not None:
    streamlit_log = open("/tmp/streamlit.log", "w")
    streamlit_proc = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", str(app_file),
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
        ],
        cwd=str(project_src),
        stdout=streamlit_log,
        stderr=subprocess.STDOUT,
    )

time.sleep(5)

# Pastikan server merespons
try:
    urllib.request.urlopen("http://127.0.0.1:8501", timeout=5)
except Exception:
    print(Path("/tmp/streamlit.log").read_text()[-5000:])
    raise

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
display(HTML(f'<a href="{url}" target="_blank">Buka aplikasi Streamlit</a>'))

In [ ]:
# Hentikan server lama
if "streamlit_proc" in globals() and streamlit_proc.poll() is None:
    streamlit_proc.terminate()
    streamlit_proc.wait(timeout=10)

# Bersihkan log lama
Path("/tmp/streamlit.log").write_text("")

# Jalankan ulang Streamlit untuk proxy Colab
streamlit_log = open("/tmp/streamlit.log", "w")
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(app_file),
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    cwd=str(project_src),
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print(Path("/tmp/streamlit.log").read_text()[-3000:])

In [ ]:
from pipeline import run_assessment

result = run_assessment(
    pdf_path="../PDF/Juknis-OMV-DNDF-Lindung-Nilai-IP.pdf",
    enable_llm_judge=False,
)

print(result["classification"], result["percentage"])


1. PARSING JUKNIS
Nama dokumen : Juknis-OMV-DNDF-Lindung-Nilai-IP.pdf
Jumlah halaman: 24

3. CHUNKING JUKNIS
Jumlah chunk: 131

4. METADATA JUKNIS
Distribusi document_type: {'JUKNIS': 131}

DEBUG EMBEDDING
Panjang karakter: 2706
Jumlah kata: 305
Preview: Bagian Dokumen: opening
Pembukaan
i

PETUNJUK TEKNIS

NOMOR 4/JUKNIS/EKSTERNAL/DPMA/2026

TENTANG
MEKANISME PELAKSANAAN TRANSAKSI DNDF LINDUNG NILAI
KEPADA BANK INDONESIA

DEPARTEMEN PENGELOLAAN MONETER DAN ASET SEKURITAS

2026
ii

KATA PENGANTAR

Dalam menetapkan dan melaksanakan kebijakan moneter,
Embedding 1/131 selesai

DEBUG EMBEDDING
Panjang karakter: 539
Jumlah kata: 34
Preview: Bagian Dokumen: main
BAB IV TRANSAKSI DNDF LINDUNG NILAI IP ............................................... 10
F.
Perpanjangan Transaksi DNDF Lindung Nilai IP .................................... 13
LAMPIRAN I ..........................................................................................
Embedding 2/131 selesai

DEBUG EMBEDDING
Panjang karak